# Prepare paired Rain13K images at 512×512

This notebook prepares Rain13K for the DCNv4 UNet and the later DiFix stage. It follows the resizing method used in `degradation_test.ipynb`: **center-crop the largest square, then resize to 512×512 with LANCZOS**.

Audience: users running the two-stage Rain13K training pipeline on the Linux server.

Prerequisites:
- The verified Rain13K pair folders and split manifests already exist.
- Pillow and tqdm are installed in the active Python environment.
- Source and output directories are on the same filesystem if hard-linked views are requested.

The notebook never overwrites the original Rain13K files. Batch processing and hard-link creation are disabled until their explicit flags are enabled.


## Workflow

1. Configure paths and safety flags.
2. Validate all rainy/clean pairs by filename stem.
3. Preview the paired center-crop and LANCZOS transform.
4. Write a resumable `Rain13K_512` dataset.
5. Validate dimensions, counts, and paired stems.
6. Build zero-copy hard-linked UNet train/validation views.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from PIL import Image
# Use the text progress bar to avoid Jupyter Widgets version conflicts.
from tqdm import tqdm

# Override RAIN13K_ROOT in the environment if the server mount changes.
DATA_ROOT = Path(os.environ.get(
    "RAIN13K_ROOT",
    "/home/bml/storage/mnt/v-zz4uoucip21b66el/PRP/Unet4Degradation/data/Rain13K",
)).expanduser()
SOURCE_ROOT = DATA_ROOT / "train" / "Rain13K"
OUTPUT_ROOT = DATA_ROOT / "train" / "Rain13K_512"
SPLIT_ROOT = DATA_ROOT / "splits"
VIEW_ROOT = DATA_ROOT / "views_512"

TARGET_SIZE = (512, 512)  # PIL uses (width, height).
EXPECTED_PAIRS = 13_711
SUPPORTED_SUFFIXES = {".jpg", ".jpeg", ".png"}

# Safety switches: inspect the validation/preview cells before enabling.
RUN_PREPROCESS = False
RUN_MATERIALIZE_VIEWS = False
OVERWRITE = False

print("Source:", SOURCE_ROOT)
print("Output:", OUTPUT_ROOT)
print("RUN_PREPROCESS:", RUN_PREPROCESS)
print("RUN_MATERIALIZE_VIEWS:", RUN_MATERIALIZE_VIEWS)


## 1. Discover and validate aligned pairs

Files are paired by stem rather than directory order. The cell refuses missing or duplicate IDs before any output is written.


In [ ]:
def files_by_stem(directory: Path) -> dict[str, Path]:
    if not directory.is_dir():
        raise FileNotFoundError(directory)
    result: dict[str, Path] = {}
    for path in directory.iterdir():
        if not path.is_file() or path.suffix.lower() not in SUPPORTED_SUFFIXES:
            continue
        if path.stem in result:
            raise ValueError(f"Duplicate stem {path.stem!r} in {directory}")
        result[path.stem] = path
    return result

def natural_id(stem: str):
    return (0, int(stem)) if stem.isdigit() else (1, stem)

def discover_pairs(source_root: Path):
    rainy = files_by_stem(source_root / "input")
    clean = files_by_stem(source_root / "target")
    if set(rainy) != set(clean):
        missing_clean = sorted(set(rainy) - set(clean), key=natural_id)[:10]
        missing_rainy = sorted(set(clean) - set(rainy), key=natural_id)[:10]
        raise ValueError(
            f"Pair mismatch: missing clean={missing_clean}, missing rainy={missing_rainy}"
        )
    stems = sorted(rainy, key=natural_id)
    return [(stem, rainy[stem], clean[stem]) for stem in stems]

if SOURCE_ROOT.is_dir():
    pairs = discover_pairs(SOURCE_ROOT)
    if len(pairs) != EXPECTED_PAIRS:
        raise ValueError(f"Expected {EXPECTED_PAIRS} pairs, found {len(pairs)}")
    print(f"Validated {len(pairs):,} paired samples")
    print("First/last IDs:", pairs[0][0], pairs[-1][0])
else:
    pairs = []
    print("Server dataset is not mounted here; function definitions remain available.")


## 2. Define and preview the paired transform

The rainy image and clean target must have identical source dimensions. A single crop box is calculated and applied to both, preserving pixel alignment.


In [ ]:
def center_square_box(size: tuple[int, int]) -> tuple[int, int, int, int]:
    width, height = size
    crop_size = min(width, height)
    left = (width - crop_size) // 2
    top = (height - crop_size) // 2
    return left, top, left + crop_size, top + crop_size

def transform_pair(rainy_path: Path, clean_path: Path):
    with Image.open(rainy_path) as rainy_source, Image.open(clean_path) as clean_source:
        rainy = rainy_source.convert("RGB")
        clean = clean_source.convert("RGB")
        if rainy.size != clean.size:
            raise ValueError(
                f"Pair size mismatch: {rainy_path.name}={rainy.size}, "
                f"{clean_path.name}={clean.size}"
            )
        box = center_square_box(rainy.size)
        rainy_512 = rainy.crop(box).resize(TARGET_SIZE, Image.Resampling.LANCZOS)
        clean_512 = clean.crop(box).resize(TARGET_SIZE, Image.Resampling.LANCZOS)
    return rainy_512, clean_512, box

if pairs:
    preview_id, preview_rainy_path, preview_clean_path = pairs[0]
    preview_rainy, preview_clean, preview_box = transform_pair(
        preview_rainy_path, preview_clean_path
    )
    print("Preview ID:", preview_id)
    print("Original size:", Image.open(preview_rainy_path).size)
    print("Crop box:", preview_box)
    print("Output sizes:", preview_rainy.size, preview_clean.size)
else:
    synthetic = Image.new("RGB", (640, 480), "gray")
    box = center_square_box(synthetic.size)
    preview = synthetic.crop(box).resize(TARGET_SIZE, Image.Resampling.LANCZOS)
    print("Synthetic smoke test:", synthetic.size, box, preview.size)


## 3. Write the complete paired dataset

Set `RUN_PREPROCESS = True` in the configuration cell, rerun from the top, and then run this cell. Existing complete pairs are validated and skipped, making the operation resumable. JPEG outputs use quality 95 and no chroma subsampling to avoid the stronger default JPEG recompression used by Pillow.


In [ ]:
def is_valid_output(path: Path) -> bool:
    if not path.is_file():
        return False
    try:
        with Image.open(path) as image:
            return image.mode == "RGB" and image.size == TARGET_SIZE
    except OSError:
        return False

def save_atomic(image: Image.Image, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f".{destination.name}.tmp")
    suffix = destination.suffix.lower()
    if suffix in {".jpg", ".jpeg"}:
        image.save(temporary, format="JPEG", quality=95, subsampling=0)
    elif suffix == ".png":
        image.save(temporary, format="PNG", compress_level=4)
    else:
        raise ValueError(f"Unsupported output suffix: {suffix}")
    temporary.replace(destination)

def preprocess_all(pairs, output_root: Path, overwrite: bool = False):
    written = 0
    skipped = 0
    for stem, rainy_path, clean_path in tqdm(pairs, desc="Center-crop + LANCZOS"):
        rainy_output = output_root / "input" / rainy_path.name
        clean_output = output_root / "target" / clean_path.name
        rainy_valid = is_valid_output(rainy_output)
        clean_valid = is_valid_output(clean_output)
        if rainy_valid and clean_valid and not overwrite:
            skipped += 1
            continue
        if not overwrite and (rainy_output.exists() or clean_output.exists()):
            raise FileExistsError(
                f"Partial or invalid output for {stem}; inspect it or set OVERWRITE=True"
            )
        rainy_512, clean_512, _ = transform_pair(rainy_path, clean_path)
        save_atomic(rainy_512, rainy_output)
        save_atomic(clean_512, clean_output)
        written += 1
    return {"written_pairs": written, "skipped_pairs": skipped}

if RUN_PREPROCESS:
    if len(pairs) != EXPECTED_PAIRS:
        raise RuntimeError("Source validation did not complete; refusing to preprocess")
    preprocess_result = preprocess_all(pairs, OUTPUT_ROOT, overwrite=OVERWRITE)
    print(preprocess_result)
else:
    print("Dry run only. Set RUN_PREPROCESS=True and rerun from the top to write files.")


## 4. Validate processed pairs

This performs the final count, filename, color-mode, and dimension checks required before training.


In [ ]:
def validate_processed(output_root: Path, expected_pairs: int):
    processed_pairs = discover_pairs(output_root)
    if len(processed_pairs) != expected_pairs:
        raise ValueError(f"Expected {expected_pairs} outputs, found {len(processed_pairs)}")
    for stem, rainy_path, clean_path in tqdm(processed_pairs, desc="Validate 512 outputs"):
        with Image.open(rainy_path) as rainy, Image.open(clean_path) as clean:
            if rainy.mode != "RGB" or clean.mode != "RGB":
                raise ValueError(f"Non-RGB output for {stem}: {rainy.mode}, {clean.mode}")
            if rainy.size != TARGET_SIZE or clean.size != TARGET_SIZE:
                raise ValueError(f"Bad output size for {stem}: {rainy.size}, {clean.size}")
    return processed_pairs

if (OUTPUT_ROOT / "input").is_dir() and (OUTPUT_ROOT / "target").is_dir():
    processed_pairs = validate_processed(OUTPUT_ROOT, EXPECTED_PAIRS)
    print(f"Validated {len(processed_pairs):,} processed pairs")
else:
    processed_pairs = []
    print("Processed directories do not exist yet. Run the preprocessing step first.")


## 5. Materialize zero-copy UNet views

After validation, set `RUN_MATERIALIZE_VIEWS = True` and rerun this cell. It reads the existing leakage-safe split lists and creates hard links for `unet_train` and `validation`. The processed source remains the single owner of the image data.


In [ ]:
def read_split_ids(path: Path) -> list[str]:
    if not path.is_file():
        raise FileNotFoundError(path)
    ids = [line.strip() for line in path.read_text().splitlines() if line.strip()]
    if len(ids) != len(set(ids)):
        raise ValueError(f"Duplicate IDs in {path}")
    return ids

def materialize_hardlink_view(split_name: str) -> int:
    ids = read_split_ids(SPLIT_ROOT / f"{split_name}.txt")
    source_maps = {
        role: files_by_stem(OUTPUT_ROOT / role) for role in ("input", "target")
    }
    for role, source_map in source_maps.items():
        destination_dir = VIEW_ROOT / split_name / role
        destination_dir.mkdir(parents=True, exist_ok=True)
        for sample_id in tqdm(ids, desc=f"Link {split_name}/{role}"):
            if sample_id not in source_map:
                raise KeyError(f"Missing processed {role} image for {sample_id}")
            source = source_map[sample_id]
            destination = destination_dir / source.name
            if destination.exists():
                if os.path.samefile(source, destination):
                    continue
                raise FileExistsError(f"Hard-link destination conflict: {destination}")
            os.link(source, destination)
    return len(ids)

if RUN_MATERIALIZE_VIEWS:
    if len(processed_pairs) != EXPECTED_PAIRS:
        raise RuntimeError("Processed dataset has not passed validation")
    expected_split_counts = {"unet_train": 8_227, "validation": 1_371}
    for split_name, expected_count in expected_split_counts.items():
        linked_count = materialize_hardlink_view(split_name)
        if linked_count != expected_count:
            raise ValueError(
                f"{split_name}: expected {expected_count} IDs, found {linked_count}"
            )
        print(f"{split_name}: {linked_count:,} linked pairs")
else:
    print("Set RUN_MATERIALIZE_VIEWS=True after processed-data validation.")


## Pitfalls and next step

- Do not process rainy and clean folders independently: one shared crop box is required for alignment.
- Do not write into the original `Rain13K` directory.
- Center cropping discards the longer image dimension. This intentionally matches the referenced notebook; use padding instead only as a separate ablation.
- Do not build views before all 13,711 processed pairs pass validation.

After completion, train the DCNv4 UNet with:
- training input: `views_512/unet_train/input`
- training GT: `views_512/unet_train/target`
- validation input: `views_512/validation/input`
- validation GT: `views_512/validation/target`

Optional exercise: compare this center-crop policy against aspect-ratio-preserving padding using the unchanged validation IDs.
